In [5]:
# Convert DICOM to NIFTI file for visualization
import numpy as np
import matplotlib.pyplot as plt
from dipy.data import fetch_tissue_data, read_tissue_data
from dipy.segment.tissue import TissueClassifierHMRF
from dipy.io.image import load_nifti
import dicom2nifti


# Input Folder with DICOM files
dicom_input = "E:\\Documents\\LP-0001-01-01-01\\LP-0001-01-01-01\\anatomicalData"
nifti_output = "E:\Documents\LP-0001-01-01-01_NIFTI"
dicom2nifti.convert_directory(dicom_input, nifti_output)




In [10]:
import nibabel as nib
from ipywidgets import interact
from matplotlib import pyplot as plt
import numpy as np

# Load Nifti File
nifti = "E:\\Documents\\N4_Test.nii"
#nifti = "E:\\Documents\\DipySegmentation\\LP-0001-01-01-01_NIFTI\\1_fl3d_ax_sel_10x10x10.nii.gz" #copy path to nifti file
image = nib.load(nifti)

# Convert the image data as a NumPy array
array = image.get_fdata()

# Rotate the image 90 degrees to the left
rotated_array = np.rot90(array, k=1, axes=(0, 1))

# Visuali ze Images as a slider
def show_slice(i):
    plt.imshow(array[:,:,i], cmap='gray')
    plt.title('Image Slice at Index {}'.format(i))
    plt.show()

interact(show_slice, i=(0, array.shape[2]-1))

interactive(children=(IntSlider(value=79, description='i', max=159), Output()), _dom_classes=('widget-interact…

<function __main__.show_slice(i)>

In [7]:
## N4 BIAS FIELD CORRECTION
raw_nifti = image

In [8]:
import SimpleITK as sitk #1
raw_img_sitk = sitk.ReadImage(nifti, sitk.sitkFloat32) #2
raw_img_sitk_arr = sitk.GetArrayFromImage(raw_img_sitk) #3

transformed = sitk.RescaleIntensity(raw_img_sitk, 0, 255) #1
transformed = sitk.LiThreshold(transformed,0,1) #2
head_mask = transformed

shrinkFactor = 4 #1
inputImage = raw_img_sitk

inputImage = sitk.Shrink( raw_img_sitk, [ shrinkFactor ] * inputImage.GetDimension() ) #2
maskImage = sitk.Shrink( head_mask, [ shrinkFactor ] * inputImage.GetDimension() ) #3

bias_corrector = sitk.N4BiasFieldCorrectionImageFilter() #4

corrected = bias_corrector.Execute(inputImage, maskImage) #5

In [9]:
print(raw_img_sitk.GetPixelIDTypeAsString())


32-bit float


In [14]:
log_bias_field = bias_corrector.GetLogBiasFieldAsImage(raw_img_sitk) #1
corrected_image_full_resolution = raw_img_sitk / sitk.Exp( log_bias_field ) #2

In [20]:
# save image
sitk.WriteImage(corrected_image_full_resolution, "E:\\Documents\\N4_Test.nii") #1